# Business Sim Env — Full Build Notebook
**Meta × PyTorch Hackathon — Round 1**

Run every cell top to bottom. Each cell writes a file using `%%writefile`.
At the end you'll have the complete project ready to push to GitHub → HF Spaces.

### Sections
1. Install deps + create folder structure
2. `src/models.py`
3. `src/environment/market_agent.py`
4. `src/environment/adversarial.py`
5. `src/environment/company_env.py`
6. Task graders
7. `src/business_sim_env.py` (client wrapper)
8. `src/server.py` (FastAPI)
9. `inference.py` (root — mandatory)
10. Config files (openenv.yaml, Dockerfile, requirements.txt, README)
11. `validate.py`
12. Smoke test
13. Export zip for GitHub

In [ ]:
# ── CELL 1: Install + scaffold ────────────────────────────────────────────────
!pip install fastapi uvicorn pydantic openai httpx pyyaml -q

import os
os.makedirs('src/environment', exist_ok=True)
os.makedirs('src/tasks',       exist_ok=True)

# __init__ files
for path in ['src/__init__.py', 'src/environment/__init__.py', 'src/tasks/__init__.py']:
    open(path, 'w').close()

print('✅ Folders and __init__ files created')

In [ ]:
%%writefile src/models.py
"""
models.py — All Pydantic typed models for the Business Sim OpenEnv environment.
Satisfies OpenEnv spec: typed models for action_space and observation_space.
"""
from __future__ import annotations
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum


class TechStack(str, Enum):
    cheap    = "cheap"
    standard = "standard"
    premium  = "premium"


class MarketPhase(str, Enum):
    boom      = "boom"
    stable    = "stable"
    recession = "recession"


class Domain(str, Enum):
    ai     = "ai"
    web    = "web"
    mobile = "mobile"
    data   = "data"
    devops = "devops"


class Project(BaseModel):
    """A client project the CEO can accept."""
    id:                 str
    name:               str
    domain:             Domain
    # Factor 1 — Profit Potential
    base_profit:        float
    profit_variance:    float
    # Factor 2 — Risk Level
    base_risk:          float
    hidden_risk:        float
    # Factor 3 — Team Capability
    skill_required:     float
    # Factor 4 — Deadline Pressure
    duration_quarters:  int
    deadline_tight:     bool
    # Factor 5 — Market Demand
    demand_sensitivity: float
    # Factor 6 — Reputation Impact
    reputation_gain:    float
    reputation_loss:    float
    # Resource cost
    resource_cost:      float


class TeamState(BaseModel):
    """Snapshot of the company's engineering team."""
    size:         int
    skill:        float
    burnout:      float
    domain_focus: Domain


class CEOAction(BaseModel):
    """
    ACTION SPACE — OpenEnv spec requires a typed action model.
    All fields have safe defaults so a no-op action is always valid.
    """
    accept_project_id: Optional[str]  = Field(default=None)
    hire_count:        int            = Field(default=0, ge=0, le=5)
    fire_count:        int            = Field(default=0, ge=0, le=3)
    training_budget:   float          = Field(default=0.0, ge=0.0, le=50_000.0)
    tech_stack:        TechStack      = Field(default=TechStack.standard)
    reduce_workload:   bool           = Field(default=False)


class CompanyObservation(BaseModel):
    """
    OBSERVATION SPACE — OpenEnv spec requires a typed observation model.
    """
    session_id:          str
    quarter:             int
    max_quarters:        int
    goal:                str
    budget:              float
    resource_pool:       float
    team:                TeamState
    reputation:          float
    market_phase:        MarketPhase
    domain_demand:       dict
    available_projects:  list[Project]
    active_risks:        list[str]
    last_action_result:  Optional[str] = None
    last_action_error:   Optional[str] = None


class StepResult(BaseModel):
    observation: CompanyObservation
    reward:      float
    done:        bool
    info:        dict


class FullState(BaseModel):
    observation:          CompanyObservation
    internal:             dict
    episode_history:      list[dict]
    counterfactual_hint:  Optional[str] = None


class TaskInfo(BaseModel):
    id:           str
    difficulty:   str
    max_quarters: int
    description:  str

In [ ]:
%%writefile src/environment/market_agent.py
"""
market_agent.py — Dynamic market environment (Factor 5: Market Demand).
Simulates real-world business cycles via Markov chain.
"""
import random
from src.models import MarketPhase

DOMAIN_DEMAND_BY_PHASE = {
    MarketPhase.boom:      {"ai": 1.4, "web": 1.1, "mobile": 1.2, "data": 1.3, "devops": 1.0},
    MarketPhase.stable:    {"ai": 1.1, "web": 1.0, "mobile": 1.0, "data": 1.0, "devops": 1.0},
    MarketPhase.recession: {"ai": 0.8, "web": 0.7, "mobile": 0.6, "data": 0.9, "devops": 1.1},
}


class MarketAgent:
    PHASES = [MarketPhase.boom, MarketPhase.stable, MarketPhase.recession]
    TRANSITION = {
        MarketPhase.boom:      [0.30, 0.50, 0.20],
        MarketPhase.stable:    [0.20, 0.50, 0.30],
        MarketPhase.recession: [0.10, 0.40, 0.50],
    }

    def __init__(self):
        self.phase = MarketPhase.stable

    def reset(self):
        self.phase = MarketPhase.stable

    def step(self) -> MarketPhase:
        self.phase = random.choices(self.PHASES, weights=self.TRANSITION[self.phase], k=1)[0]
        return self.phase

    def domain_demand(self) -> dict:
        return DOMAIN_DEMAND_BY_PHASE[self.phase].copy()

    def project_count(self) -> int:
        return {MarketPhase.boom: 4, MarketPhase.stable: 3, MarketPhase.recession: 2}[self.phase]

    def profit_multiplier(self) -> float:
        return {MarketPhase.boom: 1.25, MarketPhase.stable: 1.00, MarketPhase.recession: 0.75}[self.phase]

In [ ]:
%%writefile src/environment/adversarial.py
"""
adversarial.py — Adversarial Critic Agent (Factor 10: Uncertainty / Hidden Risk).
Injects realistic business shocks scaled by task difficulty.
"""
import random

SHOCK_CATALOG = [
    ("client_dispute",      0.12, -0.08,  -5_000),
    ("key_developer_quit",  0.10, -0.05, -10_000),
    ("budget_audit",        0.08,  0.00, -15_000),
    ("market_rumour",       0.10, -0.10,      0),
    ("competitor_launch",   0.10, -0.05,      0),
    ("surprise_tax_bill",   0.06,  0.00,  -8_000),
    ("data_breach_scare",   0.05, -0.12,  -6_000),
]

DIFFICULTY_MULTIPLIER = {"easy": 0.40, "medium": 1.00, "hard": 1.80}


class AdversarialAgent:
    def __init__(self, difficulty: str = "easy"):
        self.multiplier = DIFFICULTY_MULTIPLIER.get(difficulty, 1.0)

    def apply(self, reputation: float, budget: float) -> tuple:
        triggered = []
        for name, prob, rep_delta, budget_delta in SHOCK_CATALOG:
            if random.random() < prob * self.multiplier:
                reputation = max(0.0, min(1.0, reputation + rep_delta))
                budget    += budget_delta
                triggered.append(name)
        return reputation, budget, triggered

In [ ]:
%%writefile src/environment/company_env.py
"""
company_env.py — Core business simulation environment.
Integrates all 6 key factors + burnout/resource management.
"""
import random
import uuid

from src.models import (
    CEOAction, CompanyObservation, StepResult, FullState,
    Project, TechStack, TeamState, Domain,
)
from src.environment.market_agent import MarketAgent
from src.environment.adversarial  import AdversarialAgent


PROJECTS_POOL = [
    dict(name="AI Recommendation Engine", domain=Domain.ai,
         base_profit=90_000, profit_variance=0.40, base_risk=0.65, hidden_risk=0.25,
         skill_required=0.75, duration_quarters=2, deadline_tight=True,
         demand_sensitivity=1.4, reputation_gain=0.10, reputation_loss=0.20, resource_cost=8_000),
    dict(name="E-commerce Platform", domain=Domain.web,
         base_profit=35_000, profit_variance=0.15, base_risk=0.20, hidden_risk=0.10,
         skill_required=0.30, duration_quarters=1, deadline_tight=False,
         demand_sensitivity=0.9, reputation_gain=0.05, reputation_loss=0.08, resource_cost=2_000),
    dict(name="Mobile Fitness App", domain=Domain.mobile,
         base_profit=55_000, profit_variance=0.30, base_risk=0.40, hidden_risk=0.15,
         skill_required=0.50, duration_quarters=2, deadline_tight=False,
         demand_sensitivity=1.1, reputation_gain=0.07, reputation_loss=0.12, resource_cost=3_500),
    dict(name="Real-time Data Pipeline", domain=Domain.data,
         base_profit=65_000, profit_variance=0.25, base_risk=0.50, hidden_risk=0.20,
         skill_required=0.60, duration_quarters=1, deadline_tight=True,
         demand_sensitivity=1.2, reputation_gain=0.08, reputation_loss=0.15, resource_cost=5_000),
    dict(name="Legacy System Migration", domain=Domain.web,
         base_profit=28_000, profit_variance=0.20, base_risk=0.35, hidden_risk=0.30,
         skill_required=0.40, duration_quarters=2, deadline_tight=False,
         demand_sensitivity=0.8, reputation_gain=0.04, reputation_loss=0.10, resource_cost=1_500),
    dict(name="Cloud DevOps Pipeline", domain=Domain.devops,
         base_profit=42_000, profit_variance=0.15, base_risk=0.25, hidden_risk=0.10,
         skill_required=0.45, duration_quarters=1, deadline_tight=True,
         demand_sensitivity=1.0, reputation_gain=0.06, reputation_loss=0.09, resource_cost=4_000),
    dict(name="ML Fraud Detection API", domain=Domain.ai,
         base_profit=78_000, profit_variance=0.35, base_risk=0.60, hidden_risk=0.20,
         skill_required=0.70, duration_quarters=2, deadline_tight=True,
         demand_sensitivity=1.3, reputation_gain=0.09, reputation_loss=0.18, resource_cost=7_000),
    dict(name="Inventory SaaS Platform", domain=Domain.web,
         base_profit=32_000, profit_variance=0.10, base_risk=0.18, hidden_risk=0.08,
         skill_required=0.28, duration_quarters=1, deadline_tight=False,
         demand_sensitivity=0.85, reputation_gain=0.04, reputation_loss=0.07, resource_cost=1_000),
    dict(name="IoT Sensor Dashboard", domain=Domain.data,
         base_profit=48_000, profit_variance=0.28, base_risk=0.45, hidden_risk=0.18,
         skill_required=0.55, duration_quarters=2, deadline_tight=False,
         demand_sensitivity=1.1, reputation_gain=0.07, reputation_loss=0.13, resource_cost=4_500),
    dict(name="AR Product Visualizer", domain=Domain.mobile,
         base_profit=60_000, profit_variance=0.38, base_risk=0.55, hidden_risk=0.22,
         skill_required=0.65, duration_quarters=2, deadline_tight=True,
         demand_sensitivity=1.15, reputation_gain=0.08, reputation_loss=0.16, resource_cost=6_000),
]

TASK_GOALS = {
    "single_quarter_survival": "End the quarter with positive cash flow. Balance risk vs profit — team skill is 0.4, budget is $100k.",
    "four_quarter_growth":     "Grow company revenue by 30%% over 4 quarters. Invest in team, adapt to market shifts.",
    "adversarial_resilience":  "Survive 8 quarters with reputation >= 0.6 despite adversarial shocks. Manage burnout and hidden risks.",
}

SALARY_PER_DEV   = 8_000
HIRE_COST        = 15_000
BURNOUT_RECOVERY = 5_000


class CompanyEnv:
    def __init__(self, task_id="single_quarter_survival", max_quarters=1, difficulty="easy"):
        self.task_id      = task_id
        self.max_quarters = max_quarters
        self.difficulty   = difficulty
        self.market       = MarketAgent()
        self.adversarial  = AdversarialAgent(difficulty)
        self._session_id  = str(uuid.uuid4())
        self.history      = []
        self._reset_state()

    def reset(self):
        self._session_id = str(uuid.uuid4())
        self._reset_state()
        return _ResetResult(self._build_obs())

    def step(self, action: CEOAction) -> StepResult:
        if self.done:
            return StepResult(observation=self._build_obs(error="Episode done. Call reset()."),
                              reward=0.0, done=True, info={})
        messages = []
        reward   = 0.0
        error_msg = None
        try:
            # 1. Salary
            salary = self.team.size * SALARY_PER_DEV
            self.budget -= salary
            messages.append(f"Salaries: −${salary:,.0f}")
            # 2. Resource pool
            self.resource_pool = float(self.team.size * 100)
            # 3. Hiring
            if action.hire_count > 0:
                cost = action.hire_count * HIRE_COST
                if self.budget < cost:
                    error_msg = f"Insufficient budget to hire {action.hire_count} devs."
                    action = action.model_copy(update={"hire_count": 0})
                else:
                    self.budget += -cost
                    self.team.size += action.hire_count
                    messages.append(f"Hired {action.hire_count} (−${cost:,.0f})")
            # 4. Firing
            if action.fire_count > 0:
                fired = min(action.fire_count, self.team.size - 1)
                self.team.size = max(1, self.team.size - fired)
                messages.append(f"Released {fired} devs")
            # 5. Training
            if action.training_budget > 0:
                self.budget -= action.training_budget
                gain = min(0.15, action.training_budget / 50_000)
                self.team.skill   = min(1.0, self.team.skill + gain)
                self.team.burnout = max(0.0, self.team.burnout - 0.05)
                messages.append(f"Training: skill → {self.team.skill:.2f}")
            # 6. Burnout recovery
            if action.reduce_workload:
                self.budget -= BURNOUT_RECOVERY
                self.team.burnout = max(0.0, self.team.burnout - 0.25)
                messages.append(f"Team rested: burnout → {self.team.burnout:.2f}")
            # 7. Project
            if action.accept_project_id:
                project = self._find_project(action.accept_project_id)
                if project is None:
                    error_msg = f"Project '{action.accept_project_id}' not found."
                else:
                    ok, net, msg, rep_delta = self._execute_project(project, action.tech_stack)
                    self.budget    += net
                    self.reputation = max(0.0, min(1.0, self.reputation + rep_delta))
                    reward         += net / 100_000
                    if project.deadline_tight:
                        burn = 0.10 + self.team.burnout * 0.05
                        self.team.burnout = min(1.0, self.team.burnout + burn)
                        messages.append(f"Tight deadline: burnout → {self.team.burnout:.2f}")
                    messages.append(msg)
            # 8. Burnout penalty
            if self.team.burnout > 0.6:
                self.team.skill = max(0.05, self.team.skill - self.team.burnout * 0.03)
                reward -= 0.15
                messages.append(f"⚠ Burnout {self.team.burnout:.2f} draining skill")
            # 9. Tech debt
            if "tech_debt" in self.active_risks and random.random() < 0.4:
                self.reputation = max(0.0, self.reputation - 0.04)
                reward -= 0.05
                messages.append("Tech debt: client complaints")
            # 10. Adversarial shocks
            self.reputation, self.budget, shocks = self.adversarial.apply(self.reputation, self.budget)
            if shocks:
                messages.append(f"Shocks: {', '.join(shocks)}")
            # 11. Market advance
            self.market.step()
            # 12. Shaped reward
            reward += self.reputation * 0.20
            reward += min(0.30, self.budget / 500_000)
            reward -= self.team.burnout * 0.10
            if self.budget < 0:
                reward -= 1.0
        except Exception as exc:
            error_msg = str(exc)

        self.quarter += 1
        self.done = (self.quarter > self.max_quarters or
                     self.budget < -50_000 or
                     self.reputation <= 0.0)
        self.history.append({
            "quarter": self.quarter - 1, "action": action.model_dump(),
            "reward": round(reward, 4), "budget": round(self.budget, 2),
            "reputation": round(self.reputation, 3), "burnout": round(self.team.burnout, 3),
            "team_skill": round(self.team.skill, 3), "messages": messages,
        })
        return StepResult(
            observation=self._build_obs(last_result="; ".join(messages) or None, error=error_msg),
            reward=round(reward, 4), done=self.done,
            info={"quarter": self.quarter, "budget": round(self.budget, 2),
                  "reputation": round(self.reputation, 3)},
        )

    def get_full_state(self) -> FullState:
        hint = None
        if self.history:
            last = self.history[-1]
            if last["reward"] < 0:
                if last["burnout"] > 0.6:
                    hint = "Counterfactual: reduce_workload before that project would have prevented burnout penalty."
                elif last["action"].get("accept_project_id"):
                    hint = "Counterfactual: $20k training first would have raised success probability ~30%."
        return FullState(
            observation=self._build_obs(),
            internal={"market_phase": self.market.phase, "domain_demand": self.market.domain_demand(),
                      "active_risks": self.active_risks, "difficulty": self.difficulty},
            episode_history=self.history, counterfactual_hint=hint,
        )

    @property
    def session_id(self): return self._session_id

    def _reset_state(self):
        self.quarter = 1
        self.budget  = 100_000.0
        self.resource_pool = 500.0
        self.team    = TeamState(size=5, skill=0.40, burnout=0.0, domain_focus=Domain.web)
        self.reputation    = 0.70
        self.active_risks  = []
        self.done          = False
        self.market.reset()
        self.history       = []
        self._cached_projects = self._sample_projects()

    def _execute_project(self, project, tech):
        skill_gap       = max(0.0, project.skill_required - self.team.skill)
        burnout_penalty = self.team.burnout * 0.15
        demand          = self.market.domain_demand().get(project.domain.value, 1.0)
        revealed_hidden = project.hidden_risk * random.random()
        total_risk      = min(0.95, project.base_risk + skill_gap * 0.5 + burnout_penalty + revealed_hidden)
        profit_mult     = demand * project.demand_sensitivity
        if tech == TechStack.cheap:
            total_risk = min(0.95, total_risk + 0.15)
            profit_mult *= 0.75
            if "tech_debt" not in self.active_risks:
                self.active_risks.append("tech_debt")
        elif tech == TechStack.premium:
            total_risk = max(0.0, total_risk - 0.10)
            profit_mult *= 1.20
        success      = random.random() > total_risk
        variance     = 1.0 + random.uniform(-project.profit_variance, project.profit_variance)
        if success:
            earned    = project.base_profit * profit_mult * variance - project.resource_cost
            return True,  earned, f"✓ '{project.name}' +${earned:,.0f} (risk={total_risk:.2f})", project.reputation_gain
        else:
            penalty   = project.base_profit * 0.3 + project.resource_cost
            return False, -penalty, f"✗ '{project.name}' −${penalty:,.0f} (risk={total_risk:.2f})", -project.reputation_loss

    def _find_project(self, pid):
        return next((p for p in self._cached_projects if p.id == pid), None)

    def _sample_projects(self):
        n    = self.market.project_count()
        pool = random.sample(PROJECTS_POOL, min(n, len(PROJECTS_POOL)))
        return [Project(id=str(uuid.uuid4())[:8], **p) for p in pool]

    def _build_obs(self, last_result=None, error=None):
        self._cached_projects = self._sample_projects()
        return CompanyObservation(
            session_id=self._session_id, quarter=self.quarter, max_quarters=self.max_quarters,
            goal=TASK_GOALS.get(self.task_id, ""), budget=round(self.budget, 2),
            resource_pool=round(self.resource_pool, 1), team=self.team.model_copy(),
            reputation=round(self.reputation, 3), market_phase=self.market.phase,
            domain_demand=self.market.domain_demand(), available_projects=self._cached_projects,
            active_risks=self.active_risks.copy(), last_action_result=last_result, last_action_error=error,
        )


class _ResetResult:
    def __init__(self, obs): self.observation = obs; self.done = False; self.reward = 0.0

In [ ]:
%%writefile src/tasks/task_easy.py
"""task_easy.py — Single Quarter Survival grader."""
def grade(env) -> float:
    if env.budget <= 0:
        return round(max(0.0, 1.0 + env.budget / 50_000), 4)
    profit = env.budget - 100_000.0
    return round(min(1.0, max(0.0, profit / 50_000.0)), 4)

In [ ]:
%%writefile src/tasks/task_medium.py
"""task_medium.py — Four Quarter Growth grader."""
def grade(env) -> float:
    if not env.history: return 0.0
    growth       = (env.budget - 100_000.0) / 100_000.0
    growth_score = min(1.0, max(0.0, growth / 0.30))
    rep_score    = env.reputation
    return round(min(1.0, max(0.0, growth_score * 0.70 + rep_score * 0.30)), 4)

In [ ]:
%%writefile src/tasks/task_hard.py
"""task_hard.py — Adversarial Resilience grader."""
def grade(env) -> float:
    survived          = 1.0 if env.budget > 0 else 0.0
    profit_score      = min(1.0, max(0.0, env.budget / 200_000.0))
    reputation_score  = env.reputation
    low_burnout_score = 1.0 - env.team.burnout
    rep_target_bonus  = 0.15 if env.reputation >= 0.6 else 0.0
    raw = (survived * 0.25 + profit_score * 0.25 +
           reputation_score * 0.20 + low_burnout_score * 0.15 + rep_target_bonus)
    return round(min(1.0, max(0.0, raw)), 4)

In [ ]:
%%writefile src/tasks/__init__.py
from src.tasks import task_easy, task_medium, task_hard
__all__ = ["task_easy", "task_medium", "task_hard"]

In [ ]:
%%writefile src/business_sim_env.py
"""
business_sim_env.py — Client wrapper mirroring BrowserGymEnv pattern.
"""
import httpx
from src.models import CEOAction, CompanyObservation

class _Result:
    def __init__(self, data):
        self.observation = CompanyObservation(**data["observation"])
        self.reward = data.get("reward", 0.0)
        self.done   = data.get("done", False)
        self.info   = data.get("info", {})

class _ResetResult:
    def __init__(self, obs_data):
        self.observation = CompanyObservation(**obs_data)
        self.reward = 0.0; self.done = False; self.info = {}

class BusinessSimEnv:
    def __init__(self, base_url="http://localhost:7860", task_id="single_quarter_survival"):
        self.base_url   = base_url.rstrip("/")
        self.task_id    = task_id
        self.session_id = None
        self._client    = httpx.Client(timeout=30)

    @classmethod
    def from_docker_image(cls, image, env_vars):
        return cls(
            base_url = env_vars.get("BUSINESS_SIM_URL",  "http://localhost:7860"),
            task_id  = env_vars.get("BUSINESS_SIM_TASK", "single_quarter_survival"),
        )

    def reset(self):
        r = self._client.post(f"{self.base_url}/reset", params={"task_id": self.task_id})
        r.raise_for_status()
        data = r.json()
        self.session_id = data.get("session_id")
        return _ResetResult(data)

    def step(self, action: CEOAction):
        r = self._client.post(f"{self.base_url}/step",
                              json=action.model_dump(), params={"session_id": self.session_id})
        r.raise_for_status()
        return _Result(r.json())

    def get_full_state(self):
        r = self._client.get(f"{self.base_url}/state", params={"session_id": self.session_id})
        r.raise_for_status(); return r.json()

    def grade(self):
        r = self._client.get(f"{self.base_url}/grade", params={"session_id": self.session_id})
        r.raise_for_status(); return float(r.json()["score"])

    def close(self): self._client.close()

In [ ]:
%%writefile src/server.py
"""
server.py — FastAPI server exposing the OpenEnv standard API.
Endpoints: GET /, GET /health, GET /tasks, POST /reset, POST /step, GET /state, GET /grade
"""
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from src.models import CEOAction, StepResult, FullState
from src.environment.company_env import CompanyEnv
from src.tasks import task_easy, task_medium, task_hard

app = FastAPI(title="Business Simulation OpenEnv", version="0.1.0")

_sessions: dict[str, CompanyEnv] = {}

TASK_CONFIG = {
    "single_quarter_survival": dict(max_quarters=1, difficulty="easy",   grader=task_easy,
                                    description="End the quarter with positive cash flow."),
    "four_quarter_growth":     dict(max_quarters=4, difficulty="medium",  grader=task_medium,
                                    description="Grow revenue 30% over 4 quarters."),
    "adversarial_resilience":  dict(max_quarters=8, difficulty="hard",    grader=task_hard,
                                    description="Survive 8 quarters with reputation >= 0.6."),
}

@app.get("/")
def root(): return {"status": "ok", "env": "business-sim-env", "version": "0.1.0"}

@app.get("/health")
def health(): return {"status": "ok"}

@app.get("/tasks")
def list_tasks():
    return [{"id": tid, "difficulty": cfg["difficulty"],
             "max_quarters": cfg["max_quarters"], "description": cfg["description"]}
            for tid, cfg in TASK_CONFIG.items()]

@app.post("/reset")
def reset(task_id: str = "single_quarter_survival"):
    if task_id not in TASK_CONFIG:
        raise HTTPException(400, f"Unknown task_id '{task_id}'. Valid: {list(TASK_CONFIG)}")
    cfg = TASK_CONFIG[task_id]
    env = CompanyEnv(task_id=task_id, max_quarters=cfg["max_quarters"], difficulty=cfg["difficulty"])
    result = env.reset()
    _sessions[env.session_id] = env
    return JSONResponse(result.observation.model_dump())

@app.post("/step", response_model=StepResult)
def step(action: CEOAction, session_id: str):
    env = _req(session_id)
    if env.done: raise HTTPException(400, "Episode done. Call /reset.")
    return env.step(action)

@app.get("/state", response_model=FullState)
def state(session_id: str): return _req(session_id).get_full_state()

@app.get("/grade")
def grade(session_id: str):
    env = _req(session_id)
    score = TASK_CONFIG[env.task_id]["grader"].grade(env)
    return {"task_id": env.task_id, "score": round(float(score), 4), "done": env.done}

def _req(sid):
    if sid not in _sessions: raise HTTPException(404, "Session not found. Call /reset first.")
    return _sessions[sid]

In [ ]:
%%writefile inference.py
"""
inference.py — Business Simulation Environment
================================================
MANDATORY env vars: API_BASE_URL, MODEL_NAME, HF_TOKEN
Uses OpenAI Client for all LLM calls (mandatory per hackathon rules).
"""
import os, re, json, textwrap
from typing import List, Dict
from openai import OpenAI
from src.business_sim_env import BusinessSimEnv
from src.models import CEOAction

API_BASE_URL = os.getenv("API_BASE_URL", "https://router.huggingface.co/v1")
API_KEY      = os.getenv("HF_TOKEN") or os.getenv("API_KEY")
MODEL_NAME   = os.getenv("MODEL_NAME", "meta-llama/Llama-3.2-3B-Instruct")
ENV_URL      = os.getenv("ENV_URL", "http://localhost:7860")
MAX_STEPS    = 10
TEMPERATURE  = 0.2
MAX_TOKENS   = 300
FALLBACK_ACTION = CEOAction()
DEBUG = True

SYSTEM_PROMPT = textwrap.dedent("""
    You are the CEO of a software company. Each quarter make strategic decisions
    to maximise profit, reputation, and team health.

    KEY FACTORS every turn:
    1. PROFIT POTENTIAL    — prefer high base_profit when skill matches
    2. RISK LEVEL          — avoid projects where base_risk+hidden_risk > 0.7
    3. TEAM CAPABILITY     — never accept if skill_required > team.skill + 0.15
    4. MARKET DEMAND       — prefer domains with demand > 1.0
    5. REPUTATION IMPACT   — failure loses more rep than a win gains
    6. RESOURCE / BURNOUT  — if burnout > 0.5, set reduce_workload=true first

    Heuristics:
    - Train if team.skill < 0.5 AND budget > 40000 (training_budget: 20000)
    - Hire if team.size < 4 AND budget > 60000 (hire_count: 1)
    - Use premium for AI/data projects with base_profit > 60000
    - Use cheap ONLY if budget < 20000
    - reduce_workload=true if burnout > 0.55

    Respond ONLY with valid JSON (no markdown, no explanation):
    {
      "accept_project_id": "<id or null>",
      "hire_count": <0-3>,
      "fire_count": <0-2>,
      "training_budget": <0.0-50000.0>,
      "tech_stack": "<cheap|standard|premium>",
      "reduce_workload": <true|false>
    }
""").strip()

def build_history_lines(history: List[str]) -> str:
    return "\n".join(history[-4:]) if history else "None"

def build_user_prompt(step: int, obs, history: List[str]) -> str:
    projects_text = "\n".join(
        f"    id={p.id} | {p.name} | domain={p.domain} | profit=${p.base_profit:,.0f} "
        f"(±{p.profit_variance:.0%}) | risk={p.base_risk:.2f} hidden={p.hidden_risk:.2f} "
        f"| skill_needed={p.skill_required:.2f} | demand_sens={p.demand_sensitivity:.1f} "
        f"| rep_gain={p.reputation_gain:.2f} rep_loss={p.reputation_loss:.2f} "
        f"| deadline={'TIGHT' if p.deadline_tight else 'normal'}"
        for p in obs.available_projects
    ) or "    (none available)"
    domain_demand_text = " | ".join(f"{k}:{v:.1f}" for k, v in obs.domain_demand.items())
    return textwrap.dedent(f"""
        Step: {step} | Goal: {obs.goal}
        Quarter: {obs.quarter} / {obs.max_quarters}
        Budget: ${obs.budget:,.0f} | Resource pool: {obs.resource_pool:.0f}
        Team: {obs.team.size} devs | Skill: {obs.team.skill:.2f} | Burnout: {obs.team.burnout:.2f}
        Reputation: {obs.reputation:.2f} | Market: {obs.market_phase}
        Domain demand: {domain_demand_text}
        Active risks: {obs.active_risks or 'none'}
        Available Projects:
        {projects_text}
        Last result: {obs.last_action_result or 'none'}
        Last error:  {obs.last_action_error  or 'none'}
        Recent history:
        {build_history_lines(history)}
        Respond with ONLY the JSON object.
    """).strip()

def parse_action(response_text: str) -> CEOAction:
    if not response_text: return FALLBACK_ACTION
    try:
        clean = re.sub(r"```(?:json)?|```", "", response_text).strip()
        match = re.search(r"\{.*\}", clean, re.DOTALL)
        if not match: return FALLBACK_ACTION
        return CEOAction(**json.loads(match.group()))
    except Exception as exc:
        if DEBUG: print(f"  [parse_action] {exc}")
        return FALLBACK_ACTION

def run_task(client: OpenAI, task_id: str) -> float:
    print(f"\n{'='*60}\n  TASK: {task_id}\n{'='*60}")
    env = BusinessSimEnv.from_docker_image(
        image="business-sim-env:latest",
        env_vars={"BUSINESS_SIM_TASK": task_id, "BUSINESS_SIM_URL": ENV_URL},
    )
    history: List[str] = []
    total_reward = 0.0
    try:
        result      = env.reset()
        observation = result.observation
        print(f"  Goal: {observation.goal}")
        for step in range(1, MAX_STEPS + 1):
            if result.done: print("  Done."); break
            try:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": build_user_prompt(step, observation, history)},
                    ],
                    temperature=TEMPERATURE, max_tokens=MAX_TOKENS, stream=False,
                )
                response_text = completion.choices[0].message.content or ""
            except Exception as exc:
                print(f"  [LLM error] {exc}")
                response_text = ""
            action = parse_action(response_text)
            if DEBUG:
                print(f"  Q{step:02d} → project={action.accept_project_id} hire={action.hire_count} "
                      f"train=${action.training_budget:,.0f} stack={action.tech_stack} rest={action.reduce_workload}")
            result       = env.step(action)
            observation  = result.observation
            reward       = result.reward
            total_reward += reward
            history.append(
                f"Q{step}: pid={action.accept_project_id} stack={action.tech_stack} "
                f"→ r={reward:+.3f} budget=${observation.budget:,.0f} "
                f"rep={observation.reputation:.2f} burnout={observation.team.burnout:.2f}"
                + (" ERROR" if observation.last_action_error else "")
            )
            print(f"         r={reward:+.4f} budget=${observation.budget:,.0f} "
                  f"rep={observation.reputation:.2f} burnout={observation.team.burnout:.2f} done={result.done}")
            if result.done: print("  Episode complete."); break
        else:
            print(f"  Reached max steps ({MAX_STEPS}).")
        final_score = env.grade()
        print(f"\n  ✅ Score: {final_score:.4f} | Total Reward: {total_reward:.4f}")
        return final_score
    finally:
        env.close()

def main() -> None:
    client = OpenAI(base_url=API_BASE_URL, api_key=API_KEY)
    tasks  = ["single_quarter_survival", "four_quarter_growth", "adversarial_resilience"]
    scores: Dict[str, float] = {}
    for task_id in tasks:
        scores[task_id] = run_task(client, task_id)
    print(f"\n{'='*60}\n  FINAL SCORES\n{'='*60}")
    for tid, sc in scores.items():
        bar = '█' * int(sc * 20) + '░' * (20 - int(sc * 20))
        print(f"  {tid:<30} {sc:.4f}  {bar}")
    print(f"\n  Average: {sum(scores.values())/len(scores):.4f}\n{'='*60}")

if __name__ == "__main__":
    main()

In [ ]:
# ── Config files ──────────────────────────────────────────────────────────────

openenv_yaml = '''name: business-sim-env
version: 0.1.0
description: >
  Multi-agent business simulation where an AI CEO manages a software company
  across quarters under adversarial conditions.
tasks:
  - id: single_quarter_survival
    difficulty: easy
    max_quarters: 1
    description: End the quarter with positive cash flow.
  - id: four_quarter_growth
    difficulty: medium
    max_quarters: 4
    description: Grow revenue 30% over 4 quarters.
  - id: adversarial_resilience
    difficulty: hard
    max_quarters: 8
    description: Survive 8 quarters with reputation >= 0.6.
action_space: CEOAction
observation_space: CompanyObservation
reward:
  type: continuous
  range: [-2.0, 2.0]
  partial_progress: true
'''

dockerfile = '''FROM python:3.11-slim
RUN useradd -m -u 1000 appuser
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
ENV PYTHONPATH=/app
EXPOSE 7860
USER appuser
CMD ["uvicorn", "src.server:app", "--host", "0.0.0.0", "--port", "7860"]
'''

requirements = '''fastapi==0.111.0
uvicorn==0.29.0
pydantic==2.7.0
openai==1.30.0
httpx==0.27.0
pyyaml==6.0.1
'''

gitignore = '''__pycache__/
*.py[cod]
.env
.venv/
venv/
'''

env_example = '''API_BASE_URL=https://router.huggingface.co/v1
MODEL_NAME=meta-llama/Llama-3.2-3B-Instruct
HF_TOKEN=your_hf_token_here
ENV_URL=http://localhost:7860
'''

for fname, content in [
    ('openenv.yaml',    openenv_yaml),
    ('Dockerfile',      dockerfile),
    ('requirements.txt',requirements),
    ('.gitignore',      gitignore),
    ('.env.example',    env_example),
]:
    with open(fname, 'w') as f:
        f.write(content)
    print(f'✅ wrote {fname}')

In [ ]:
# ── Smoke test (no server needed) ─────────────────────────────────────────────
import sys
sys.path.insert(0, '.')

from src.models import CEOAction, TechStack
from src.environment.company_env import CompanyEnv
from src.tasks import task_easy, task_medium, task_hard

print('── Smoke test ──────────────────────────────────────')

# Test all 3 envs + graders
configs = [
    ('single_quarter_survival', 1, 'easy',   task_easy),
    ('four_quarter_growth',     4, 'medium',  task_medium),
    ('adversarial_resilience',  8, 'hard',    task_hard),
]

for task_id, max_q, diff, grader in configs:
    env    = CompanyEnv(task_id=task_id, max_quarters=max_q, difficulty=diff)
    result = env.reset()
    obs    = result.observation
    
    # Run 2 steps
    for _ in range(2):
        if env.done: break
        # Accept first project if available, else no-op
        pid    = obs.available_projects[0].id if obs.available_projects else None
        action = CEOAction(accept_project_id=pid, tech_stack=TechStack.standard)
        sr     = env.step(action)
        obs    = sr.observation
    
    score = grader.grade(env)
    state = env.get_full_state()
    
    assert 0.0 <= score <= 1.0, f'Score out of range: {score}'
    assert len(state.episode_history) > 0
    print(f'  ✅ {task_id:<30} score={score:.4f} history={len(state.episode_history)} steps')

print()
print('ALL SMOKE TESTS PASSED ✅')
print('Next step: run the server and run validate.py')

In [ ]:
# ── Export zip for GitHub ─────────────────────────────────────────────────────
import zipfile, os

EXCLUDE = {'__pycache__', '.ipynb_checkpoints', '.git'}

with zipfile.ZipFile('business-sim-env.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk('.'):
        dirs[:] = [d for d in dirs if d not in EXCLUDE]
        for file in files:
            if file.endswith('.pyc'): continue
            path = os.path.join(root, file)
            zf.write(path, path)

size = os.path.getsize('business-sim-env.zip') / 1024
print(f'✅ business-sim-env.zip created ({size:.1f} KB)')
print()
print('Next steps:')
print('  1. Download business-sim-env.zip')
print('  2. Unzip and: git init && git add . && git commit -m "initial"')
print('  3. Create HF Space (Docker) at huggingface.co/new-space')
print('  4. git remote add hf https://huggingface.co/spaces/USERNAME/business-sim-env')
print('  5. git push hf main')
print('  6. Watch build logs → copy URL → submit')